# 00 — Schema Probe

Thin notebook (per PLAN.md §6): calls into `src/cragb`, makes no
pipeline decisions of its own. Purpose: confirm the raw JSONL(.gz) files
can be streamed safely and pin down their actual schema (field names,
dtypes, null rates) *before* T1.4 writes any cleaning/filtering logic
against assumed field names.

In [2]:
import sys
sys.path.insert(0, "../src")

from cragb.data.load import stream_records, sample_dataframe, schema_summary, count_records
from cragb.utils.io import load_config

cfg = load_config("configs/data.yaml")  # resolve_path anchors to repo root, not notebook cwd
cfg["paths"]

{'raw_dir': 'data/raw',
 'processed_dir': 'data/processed',
 'reviews_raw': 'data/raw/reviews_clothing.jsonl.gz',
 'metadata_raw': 'data/raw/meta_clothing.jsonl.gz',
 'raw_checksums': 'data/raw/checksums.json',
 'corpus_out': 'data/processed/corpus_v1.parquet',
 'manifest_out': 'data/processed/corpus_v1_manifest.json'}

## Reviews file — first few raw records

In [3]:
import itertools

first_records = list(itertools.islice(stream_records(cfg["paths"]["reviews_raw"]), 3))
for r in first_records:
    print({k: (v if not isinstance(v, str) else v[:80]) for k, v in r.items()})
    print("---")

{'rating': 3.0, 'title': 'Arrived Damaged : liquid in hub locker!', 'text': 'Unfortunately Amazon in their wisdom (cough, cough) decided to ship the snowsuit', 'images': [{'small_image_url': 'https://m.media-amazon.com/images/I/710WrjJi+hL._SL256_.jpg', 'medium_image_url': 'https://m.media-amazon.com/images/I/710WrjJi+hL._SL800_.jpg', 'large_image_url': 'https://m.media-amazon.com/images/I/710WrjJi+hL._SL1600_.jpg', 'attachment_type': 'IMAGE'}, {'small_image_url': 'https://m.media-amazon.com/images/I/712+yNg8COL._SL256_.jpg', 'medium_image_url': 'https://m.media-amazon.com/images/I/712+yNg8COL._SL800_.jpg', 'large_image_url': 'https://m.media-amazon.com/images/I/712+yNg8COL._SL1600_.jpg', 'attachment_type': 'IMAGE'}, {'small_image_url': 'https://m.media-amazon.com/images/I/7132ByALa+L._SL256_.jpg', 'medium_image_url': 'https://m.media-amazon.com/images/I/7132ByALa+L._SL800_.jpg', 'large_image_url': 'https://m.media-amazon.com/images/I/7132ByALa+L._SL1600_.jpg', 'attachment_type': 'IMAG

## Reviews — schema summary on a bounded sample

`sample_dataframe` only materializes `SAMPLE_N` records, never the full multi-GB file.

In [4]:
SAMPLE_N = 20_000

reviews_sample = sample_dataframe(cfg["paths"]["reviews_raw"], SAMPLE_N)
print(reviews_sample.shape)
schema_summary(reviews_sample)

(20000, 10)


,dtype,null_count,null_pct,example
column,,,,
rating,float64,0,0.0,3.0
title,object,0,0.0,'Arrived Damaged : liquid in hub locker!'
text,object,0,0.0,"'Unfortunately Amazon in their wisdom (cough, ..."
images,object,0,0.0,[{'small_image_url': 'https://m.media-amazon.c...
asin,object,0,0.0,'B096S6LZV4'
parent_asin,object,0,0.0,'B09NSZ5QMF'
user_id,object,0,0.0,'AFKZENTNBQ7A7V7UXW5JJI6UGRYQ'
timestamp,int64,0,0.0,1677938767351
helpful_vote,int64,0,0.0,0


## Metadata file — first few raw records

In [5]:
first_meta = list(itertools.islice(stream_records(cfg["paths"]["metadata_raw"]), 3))
for r in first_meta:
    print({k: (v if not isinstance(v, str) else v[:80]) for k, v in r.items()})
    print("---")

{'main_category': 'AMAZON FASHION', 'title': "BALEAF Women's Long Sleeve Zip Beach Coverup UPF 50+ Sun Protection Hooded Cover", 'average_rating': 4.2, 'rating_number': 422, 'features': ['90% Polyester, 10% Spandex', 'Zipper closure', 'Machine Wash', 'Long sleeve sun protection coverups--UPF 50+ blocks the sun from burning', 'Zipped v-neckline--fashionable V neck and smooth 1/4 zipper allows to staying place as you like', 'Two drop-in side pockets--hold your phone or keys well，no worries of falling out', 'Hoodie with non-slip drawcord--Enhancing hooded design is convenient to wrap your face and enough space to put your head and hair easily', 'A flattering coverups company you spend all day on the beach，traveling with lovers or busying around house. Recommended For everyday leisure or daily exercise'], 'description': [], 'price': 31.99, 'images': [{'thumb': 'https://m.media-amazon.com/images/I/31zPYU+UkVL._AC_SR38,50_.jpg', 'large': 'https://m.media-amazon.com/images/I/31zPYU+UkVL._AC_.

## Metadata — schema summary on a bounded sample

In [6]:
metadata_sample = sample_dataframe(cfg["paths"]["metadata_raw"], SAMPLE_N)
print(metadata_sample.shape)
schema_summary(metadata_sample)

(20000, 16)


,dtype,null_count,null_pct,example
column,,,,
main_category,object,288,1.44,'AMAZON FASHION'
title,object,0,0.00,"""BALEAF Women's Long Sleeve Zip Beach Coverup ..."
average_rating,float64,0,0.00,4.2
rating_number,int64,0,0.00,422
features,object,0,0.00,"['90% Polyester, 10% Spandex', 'Zipper closure..."
description,object,0,0.00,[]
price,float64,9154,45.77,31.99
images,object,0,0.00,[{'thumb': 'https://m.media-amazon.com/images/...
videos,object,0,0.00,"[{'title': ""Women's UPF 50+ Front Zip Beach Co..."


## Memory-safety check

Confirms `stream_records` holds flat memory while advancing through the
file, i.e. it never buffers the whole file — the property T1.4/T1.5's
streamed filters and joins depend on.

In [7]:
import os
import psutil

proc = psutil.Process(os.getpid())
rss_start_mb = proc.memory_info().rss / 1e6

n = 0
checkpoints = []
for _ in stream_records(cfg["paths"]["reviews_raw"]):
    n += 1
    if n % 100_000 == 0:
        checkpoints.append((n, proc.memory_info().rss / 1e6))
    if n >= 300_000:
        break

print(f"start RSS: {rss_start_mb:.1f} MB")
for count, rss in checkpoints:
    print(f"{count:>7,} records -> RSS {rss:.1f} MB")
assert max(rss for _, rss in checkpoints) - rss_start_mb < 200, "RSS grew >200MB while streaming — investigate a leak"
print("OK: RSS stayed flat while streaming 300k records")

start RSS: 393.3 MB
100,000 records -> RSS 393.3 MB
200,000 records -> RSS 393.3 MB
300,000 records -> RSS 393.3 MB
OK: RSS stayed flat while streaming 300k records


## Row-count sanity check (streamed, single pass)

In [8]:
# NOTE: this walks the entire multi-GB file once; it is here for the
# record but is slow (~minutes). Skip by setting RUN_FULL_COUNT = False.
RUN_FULL_COUNT = True

if RUN_FULL_COUNT:
    n_reviews = count_records(cfg["paths"]["reviews_raw"])
    n_meta = count_records(cfg["paths"]["metadata_raw"])
    print(f"reviews: {n_reviews:,} rows")
    print(f"metadata: {n_meta:,} rows")
else:
    print("Skipped full count (RUN_FULL_COUNT=False) — sample-based schema checks above are the T1.3 deliverable.")

reviews: 66,033,346 rows
metadata: 7,218,481 rows


         token_len      n_images
count  20000.00000  20000.000000
mean      80.69375      0.188100
std      101.88717      0.737798
min        0.00000      0.000000
25%       18.00000      0.000000
50%       47.00000      0.000000
75%      107.00000      0.000000
max     1203.00000     13.000000
image coverage %: 9.34
